# SLM/LLM Safety Guard Pilot Evaluation Notebook (Colab / GPU)

This notebook provides the GPU execution environment for running safety guard models (WebGuard, Llama Guard, Qwen3Guard, DynaGuard, PolicyGuard) and Large Model Judges on matched-pair contextual test cases.

### Cell 1 — Project Repository & Path Setup

In [ ]:
import os
import sys

# If running in Google Colab, set project working directory
if os.path.exists('/content'):
    %cd /content
    if not os.path.exists('slm-guard-pilot'):
        !git clone https://github.com/your-repo/slm-guard-pilot.git
    %cd slm-guard-pilot

sys.path.insert(0, os.getcwd())
print(f"Current working directory: {os.getcwd()}")

### Cell 2 — Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q transformers torch accelerate bitsandbytes

### Cell 3 — Verify GPU Hardware & VRAM

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Running in CPU mode. (Note: Real LLMs require a GPU runtime in Colab: Runtime -> Change runtime type -> T4/A100 GPU).")

### Cell 4 — Load Test Cases

In [ ]:
from src.data.loader import load_test_cases

dataset_path = 'data/dummy/dummy_cases.json'
cases = load_test_cases(dataset_path)
print(f"Loaded {len(cases)} test cases across {len(set(c.pair_id for c in cases))} matched pairs.")

### Cell 5 — Validate Test Cases Against JSON Schema

In [ ]:
from src.data.validator import validate_dataset_file

is_valid, errors = validate_dataset_file(dataset_path)
if is_valid:
    print("SUCCESS: Dataset schema validation passed!")
else:
    print(f"Validation errors ({len(errors)}):", errors)

### Cell 6 — Run CPU Mock Model Baseline

In [ ]:
from src.models.mock_guard import MockGuard
from src.baselines.rule_baseline import RuleBaseline
from src.evaluation.runner import run_evaluation_for_model
from src.evaluation.metrics import calculate_overall_metrics
from src.evaluation.pair_metrics import calculate_pair_metrics

mock_model = MockGuard(name="mock_guard", config={"mode": "perfect"})
results = run_evaluation_for_model(mock_model, cases)

overall = calculate_overall_metrics(results)
pair_m = calculate_pair_metrics(results)

print(f"Mock Accuracy: {overall['overall_accuracy']*100:.1f}%")
print(f"Mock Pair Accuracy: {pair_m['pair_accuracy']*100:.1f}%")
print(f"Mock Context Flip Rate: {pair_m['context_flip_rate']*100:.1f}%")

### Cell 7 — Run GPU Safety Guard Model (e.g., WebGuard / QwenGuard)

In [ ]:
from src.models.qwen_guard import QwenGuardAdapter

# Configure Hugging Face checkpoint in models.yaml or dynamically here
qwen_config = {"enabled": True, "checkpoint": "Qwen/Qwen2.5-7B-Instruct"}
qwen_guard = QwenGuardAdapter(name="qwen_guard", config=qwen_config)

try:
    guard_results = run_evaluation_for_model(qwen_guard, cases)
except NotImplementedError as e:
    print("Guard adapter configuration note:", e)

### Cell 8 — Calculate Comprehensive Metrics

In [ ]:
from src.evaluation.failure_analysis import analyze_failures

failures = analyze_failures(results)
print(f"Total failures: {failures['total_failures']}")
print(f"Error breakdown: {failures['error_type_counts']}")

### Cell 9 — Generate Decision Gate Report

In [ ]:
from src.evaluation.decision_gate import generate_decision_gate_report

report_path = 'reports/colab_decision_gate_report.md'
report_md = generate_decision_gate_report(overall, pair_m, failures, report_path)
print(f"Generated report at {report_path}")